# Dynamic Data Augmentation

Dynamic data augmentation applies transformations to images during training rather than beforehand. Unlike static augmentation—such as techniques used for dataset balancing—where modified images are preprocessed and stored, dynamic augmentation generates new variations in each training iteration. This approach improves model generalization by exposing it to a wider range of variations, reducing overfitting, and enhancing robustness.

We will explore two levels of data augmentation:

- **Without increasing the dataset size**: Each image undergoes random transformations in every epoch, ensuring the model never sees the exact same image twice.

- **Expanding the dataset**: Multiple transformations are applied to each image, effectively increasing the dataset size by including different augmented versions.

In this notebook, we will apply some of the transformations explored in the previous notebook.

In [ ]:
import torch
import torchvision
from torchvision.transforms import v2

import torchinfo

import matplotlib.pyplot as plt
import numpy as np
import time

# importing a module with utilities for displaying stats and data
import sys
sys.path.insert(1, 'util')
import vcpi_util

## Auxiliary functions from previous classes

In [ ]:
def train_III(model, train_loader, val_loader, epochs, loss_fn, optimizer, scheduler, early_stopper, save_prefix = 'model'):

    history = {}
    history['accuracy'] = []
    history['val_acc'] = []
    history['val_loss'] = []
    history['loss'] = []
    best_val_loss = np.inf

    for epoch in range(epochs):  # loop over the dataset multiple times

        model.train()
        start_time = time.time() 
        correct = 0
        running_loss = 0.0
        for i, (inputs, targets) in enumerate(train_loader, 0):
            
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            loss = loss_fn(outputs, targets)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss
            correct += (predicted == targets).sum()

        model.eval()
        v_correct = 0
        val_loss = 0.0
        with torch.no_grad():
            for i,t in val_loader:
                i = i.to(device)
                t = t.to(device)
                o = model(i)
                _,p = torch.max(o,1)
                
                #with torch.no_grad():
                val_loss += loss_fn(o, t)

                v_correct += (p == t).sum()

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        
        if old_lr != new_lr:
            print('==> Learning rate updated: ', old_lr, ' -> ', new_lr)

        epoch_loss = running_loss / len(train_loader.dataset)
        accuracy = 100 * correct / len(train_loader.dataset)
        v_accuracy = 100 * v_correct / len(val_loader.dataset)
        val_loss = val_loss / len(val_loader.dataset)
        stop_time = time.time()
        print(f'Epoch: {epoch:03d}; Loss: {epoch_loss:0.6f}; Accuracy: {accuracy:0.4f}; Val Loss: {val_loss:0.6f}; Val Acc: {v_accuracy:0.4f}; Elapsed time: {(stop_time - start_time):0.4f}')
        history['accuracy'].append(accuracy.cpu().numpy())
        history['val_acc'].append(v_accuracy.cpu().numpy())
        history['val_loss'].append(val_loss.cpu().detach().numpy())
        history['loss'].append(epoch_loss.cpu().detach().numpy())
 
        ###### Saving ######
        if val_loss < best_val_loss:
           
            torch.save({
                'epoch': epoch,
                'model':model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict()
                },
                f'{save_prefix}_best.pt')

        if early_stopper(val_loss):
            print('Early stopping!')
            break
        
    print('Finished Training')

    return(history)


def evaluate(model, data_loader):

    # sets the model in evaluation mode.
    # although our model does not have layers which behave differently during training and evaluation
    # this is a good practice as the models architecture may change in the future
    model.eval()

    correct = 0
    
    for i, (images, targets) in enumerate(data_loader):
         
        # forward pass, compute the output of the model for the current batch
        outputs = model(images.to(device))

        # "max" returns a namedtuple (values, indices) where values is the maximum 
        # value of each row of the input tensor in the given dimension dim; 
        # indices is the index location of each maximum value found (argmax).
        # the argmax effectively provides the predicted class number        
        _, preds = torch.max(outputs, dim=1)

        correct += (preds.cpu() == targets).sum()

    return (correct / len(data_loader.dataset)).item()


class Conv(torch.nn.Module):

    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, 3)
        self.bn1 = torch.nn.BatchNorm2d(16)
        self.relu1 = torch.nn.ReLU()

        self.conv2 = torch.nn.Conv2d(16, 32, 3)
        self.bn2 = torch.nn.BatchNorm2d(32)
        self.relu2 = torch.nn.ReLU()

        self.maxpool1 = torch.nn.MaxPool2d(2)


        self.conv3 = torch.nn.Conv2d(32, 48, 3)
        self.bn3 = torch.nn.BatchNorm2d(48)
        self.relu3 = torch.nn.ReLU()

        self.conv4 = torch.nn.Conv2d(48, 48, 3)
        self.bn4 = torch.nn.BatchNorm2d(48)
        self.relu4 = torch.nn.ReLU()

        self.maxpool2 = torch.nn.MaxPool2d(2)

        self.fc1 = torch.nn.Linear(1200, num_classes)
        

    def forward(self, x):    
        
        # input = (bs, 3, 32, 32)
        x = self.conv1(x) # -> (bs, 16, 26, 26)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.conv2(x) # -> (bs, 32, 24, 24)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.maxpool1(x)
        
        x = self.conv3(x) # -> (bs, 48, 22, 22)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.conv4(x) # -> (bs, 48, 20, 20)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.maxpool2(x)
        
        x = torch.flatten(x,1) # -> (bs, 48 * 20 * 20 = 1920)
        x = self.fc1(x)        # -> (bs, 10)

        return(x)



class Early_Stopping():

    def __init__(self, patience = 3, min_delta = 0.00001):

        self.patience = patience 
        self.min_delta = min_delta

        self.min_delta
        self.min_val_loss = float('inf')

    def __call__(self, val_loss):

        # improvement
        if val_loss + self.min_delta < self.min_val_loss:
            self.min_val_loss = val_loss
            self.counter = 0

        # no improvement            
        else:
            self.counter += 1
            if self.counter > self.patience:
                return True
            
        return False
    


def build_confusion_matrix(model, dataset):

    preds = []
    ground_truth = []

    for images, targets in dataset:

        predictions = model(images.to(device))
        preds_sparse = [np.argmax(x) for x in predictions.cpu().detach().numpy()]
        preds.extend(preds_sparse)
        ground_truth.extend(targets.numpy())

    vcpi_util.show_confusion_matrix(ground_truth, preds, len(test_set.classes))      


## Settings

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

BASE_PATH = 'd:/vcpi/gtsrb'

PATH_TRAINING_SET = f'{BASE_PATH}/train_balanced_2000'
PATH_TEST_SET = f'{BASE_PATH}/test_A'
PATH_VAL_SET = f'{BASE_PATH}/val'

BATCH_SIZE = 32

EPOCHS = 50

RUNS = 5

In [ ]:
transform = v2.Compose(
    [v2.Resize((32,32)), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)]) 

# No shuffle is required for the test set, also the batch size can be completely different
test_set = torchvision.datasets.ImageFolder(root=PATH_TEST_SET, transform = transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size = BATCH_SIZE)

val_set = torchvision.datasets.ImageFolder(root=PATH_VAL_SET, transform = transform)
val_loader = torch.utils.data.DataLoader(val_set, batch_size = BATCH_SIZE)

train_set = torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform = transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size = BATCH_SIZE, shuffle=True)

## Previous accuracy performance

show accuracies for models trained with data augmentation

In [ ]:
res = 0
model_Conv = Conv(len(train_set.classes))
model_Conv.to(device)

for i in range(RUNS):

    reload = torch.load(f'gtsrb_balanced_{i}_best.pt')
    model_Conv.load_state_dict(reload['model'])
    eval = evaluate(model_Conv, test_loader)
    eval_val = evaluate(model_Conv, val_loader)
    print(reload['epoch'], eval, eval_val)    
    res += eval

res /= RUNS    
print(res)

## Data augmentation
 
### Setting up a transform for dynamic data augmentation

In [ ]:
transform_aug = v2.Compose(
    [v2.Resize((32,32)), 
     v2.RandomAffine(degrees=10, translate = [0.05, 0.05], interpolation =v2.InterpolationMode.BILINEAR), 
     v2.ColorJitter(brightness = 0.5, contrast = 0.5,hue = 0.1),
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)]) 

train_set = torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform = transform_aug)
train_loader = torch.utils.data.DataLoader(train_set, batch_size = BATCH_SIZE, shuffle=True)

### Training

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

for i in range(RUNS):
    model_Conv_II = Conv(len(train_set.classes))
    model_Conv_II.to(device)

    early_stop = Early_Stopping(9)
    optimizer_II = torch.optim.Adam(model_Conv_II.parameters())
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_II, mode='min', factor = 0.1, patience=3)    
    history_II = train_III(model_Conv_II, train_loader, val_loader, EPOCHS, loss_fn, optimizer_II, scheduler, early_stop, f'aug_I_{i}')

### Evaluating

In [ ]:
res = 0
model_Conv = Conv(len(train_set.classes))
model_Conv.to(device)

for i in range(RUNS):

    reload = torch.load(f'aug_I_{i}_best.pt')
    model_Conv.load_state_dict(reload['model'])
    eval = evaluate(model_Conv, test_loader)
    eval_val = evaluate(model_Conv, val_loader)
    print(reload['epoch'], eval, eval_val)
    res += eval

res /= RUNS    
print(res)

## Massive Data Augmentation

Here, we will significantly expand the dataset by applying multiple transformations in combination.

### Setting the transform

In [ ]:
transform_A = v2.Compose(
    [v2.Resize((32,32)), 
     v2.RandomAffine(degrees=10,interpolation =v2.InterpolationMode.BILINEAR), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)]) 

transform_B = v2.Compose(
    [v2.Resize((32,32)), 
     v2.RandomAffine(degrees=0, translate = [0.05, 0.05],interpolation =v2.InterpolationMode.BILINEAR), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)]) 

transform_C = v2.Compose(
     [v2.Resize((32,32)), 
     v2.ColorJitter(brightness = 0.5), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)])

transform_D = v2.Compose(
     [v2.Resize((32,32)), 
     v2.ColorJitter(contrast = 0.5), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)])

transform_E = v2.Compose(
     [v2.Resize((32,32)), 
     v2.ColorJitter(hue = 0.1), 
     v2.ToImage(), 
     v2.ToDtype(torch.float32, scale=True)])    



train_loader_ = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([
        
        torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform=transform_A),
        torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform=transform_B),
        torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform=transform_C),
        torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform=transform_D),
        torchvision.datasets.ImageFolder(root=PATH_TRAINING_SET, transform=transform_E),

    ]),  batch_size = BATCH_SIZE, shuffle = True)

### Training

In [ ]:

for i in range(RUNS):
    model_Conv_III = Conv(len(train_set.classes))
    model_Conv_III.to(device)

    optimizer_III = torch.optim.Adam(model_Conv_III.parameters())
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_III, mode='min', factor = 0.1, patience=3)
    loss_fn = torch.nn.CrossEntropyLoss()

    early_stop = Early_Stopping(9)
    history_III = train_III(model_Conv_III, train_loader_, val_loader, EPOCHS, loss_fn, optimizer_III, scheduler, early_stop, f'aug_II_{i}')

### Evaluating

In [ ]:
res = 0
model_Conv = Conv(len(train_set.classes))
model_Conv.to(device)

for i in range(RUNS):

    reload = torch.load(f'aug_II_{i}_best.pt')
    model_Conv.load_state_dict(reload['model'])
    eval = evaluate(model_Conv, test_loader)
    eval_val = evaluate(model_Conv, val_loader)
    print(reload['epoch'], eval, eval_val)    
    res += eval

res /= RUNS    
print(res)